## 1. Import Libraries
Import core libraries required for data handling (pandas, numpy) and visualization (matplotlib).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Basic Data Audit
This cell checks dataset shape, column names, data types, missing values, and target balance.

In [4]:
df = pd.read_csv("../train.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
0,641,53,Male,Switzerland,non-anginal,160.0,0.0,NaN,lv hypertrophy,122.0,True,0.0,NaN,NaN,reversable defect,1
1,744,74,Male,VA Long Beach,non-anginal,NaN,0.0,False,normal,NaN,NaN,NaN,NaN,NaN,NaN,0
2,891,53,Male,VA Long Beach,asymptomatic,124.0,243.0,False,normal,122.0,True,2.0,flat,NaN,reversable defect,1
3,271,61,Male,Cleveland,asymptomatic,140.0,207.0,False,lv hypertrophy,138.0,True,1.9,upsloping,1.0,reversable defect,1
4,655,56,Male,Switzerland,non-anginal,155.0,0.0,False,st-t abnormality,99.0,False,0.0,flat,NaN,normal,1


## 3. Basic Data Audit
Inspect dataset structure: shape, column names, data types, missing values, and target distribution to assess class imbalance.

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget counts:")
print(df["target"].value_counts())

print("\nTarget proportions:")
print(df["target"].value_counts(normalize=True))

## 4. Target Distribution Visualization
Visualize the distribution of the target variable (heart disease vs no disease) to support evaluation metric selection.

In [ ]:
df["target"].value_counts().plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Target (0 = No Disease, 1 = Disease)")
plt.ylabel("Count")
plt.show()

## 5. Institutional (Dataset) Analysis
Analyze distribution of patients and disease prevalence across contributing hospitals using the `dataset` column.

In [ ]:
print(df["dataset"].value_counts())

site_target = pd.crosstab(df["dataset"], df["target"])
print(site_target)

site_target.plot(kind="bar", stacked=True)
plt.title("Heart Disease Distribution by Hospital")
plt.xlabel("Hospital")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Data Quality Check (Invalid Values)
Identify biologically implausible values (e.g., cholesterol = 0) which indicate missing or corrupted data.

In [ ]:
print("chol == 0:", (df["chol"] == 0).sum())
print("trestbps == 0:", (df["trestbps"] == 0).sum())

if "thalch" in df.columns:
    print("thalch == 0:", (df["thalch"] == 0).sum())

## 7. Replace Invalid Values
Convert invalid placeholder values (e.g., zeros) into proper missing values (NaN) for correct downstream handling.

In [ ]:
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)

if "thalch" in df.columns:
    df["thalch"] = df["thalch"].replace(0, np.nan)

## 8. Feature Type Identification
Separate numerical and categorical features and exclude non-predictive columns (`id`, `target`) from preprocessing groups.

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "string", "bool"]).columns.tolist()

for col in ["id", "target"]:
    if col in num_cols:
        num_cols.remove(col)
    if col in cat_cols:
        cat_cols.remove(col)

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

## 9. Missing Value Imputation
Fill missing values using:
- Median for numerical features
- Mode for categorical features
Ensure no missing values remain in modelling features.

In [ ]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

## 10. Remove Identifier Column
Drop the `id` column to prevent data leakage, as it does not carry predictive clinical information.

In [ ]:
if "id" in df.columns:
    df = df.drop(columns=["id"])

print("id still present?", "id" in df.columns)

## 11. Categorical Feature Encoding
Convert categorical variables into numerical format using one-hot encoding to make them usable by machine learning models.

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True,
    prefix_sep="_"
)

print("Encoded shape:", df_encoded.shape)
df_encoded.head()

## 12. Final Dataset Preparation
Verify no remaining missing values and split the dataset into:
- Features (X)
- Target variable (y)
Ready for model training.

In [ ]:
print("Remaining missing values after preprocessing:")
print(df_encoded.isnull().sum().sum())

X = df_encoded.drop("target", axis=1)
y = df_encoded["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Task 1 Summary

1. Loaded and inspected the dataset structure and contents.
2. Assessed class balance and justified use of F1-score.
3. Analyzed institutional distribution using the `dataset` feature.
4. Identified invalid biological values in key clinical variables.
5. Converted invalid values to missing values.
6. Defined numerical and categorical feature groups.
7. Imputed missing values using appropriate statistical methods.
8. Removed the `id` column to prevent data leakage.
9. Encoded categorical variables using one-hot encoding.
10. Prepared a clean, fully numeric dataset with no missing values.
11. Generated model-ready features (X) and target (y).